# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

In [ ]:
%load_ext autoreload
%autoreload 2

from promptpotter.presentation.ui.campaign import *

# --- Services ---
session = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

# --- Campaign config ---
# ALL experiment knobs live here — no hidden defaults in service code.
campaign_config = {
    "eval_sample_size": 15,  # queries per eval step (0 = all)
    "exclude_nodes": ["llm_ranking"],  # nodes to skip (e.g. ["entity_profiling"])
    # --- Backend node overrides ---
    # Override values from GET /pipeline. Nested format: {"node": {"param": value}}.
    # See show_pipeline_snapshot() output for available params per node.
    "pipeline_overrides": {
        "web_search": {
            "max_sites": 20,
            "num_results": 20,
            "content_char_limit": 800,
        },
        "entity_profiling": {
            "model": "openai/gpt-oss-120b",
            "max_tokens": 4000,
        },
        "llm_ranking": {
            "model": "openai/gpt-oss-120b",
        },
    },
    "optimization": {
        # --- Core loop ---
        "l1_patience": 2,  # consecutive non-improvements before stop/escalate
        "max_rounds": None,  # None = unlimited
        "n_variants": 5,  # candidates per round
        "creativity": 0.7,  # temperature for candidate generation
        "improvement_threshold": 0.01,  # accuracy delta to count as improvement
        "seed": 42,  # subsampling seed (reproducibility)
        "max_failures": 15,  # failure examples fed to LLM candidate generation
        # --- Escalation ---
        "degradation_threshold": 0.4,  # fraction of degraded queries to trigger escalation
        "backend_warning_threshold": 2,  # degradation resets before backend advisory
        "enable_l2": True,  # L2 refine_strategy on escalation
        "enable_l3": True,  # L3 modify_plan on L2 stall
        "l2_patience": 2,  # L2 stalls before L3
        "l3_patience": 1,  # L3 stalls before stop
        "l2_temperature": 0.3,  # LLM temperature for L2 transitions
        "l3_temperature": 0.5,  # LLM temperature for L3 transitions
        # --- Critique ---
        "enable_critique": True,  # critique agent between generate/evaluate
    },
    "optimizer_llm": {
        # "model":       "moonshotai/kimi-k2-instruct-0905",  # 10x more expensive
        "model": "openai/gpt-oss-120b",
        "provider": "groq",
        "temperature": 0.4,
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",
        # "model": "claude-sonnet-4-6",
        # "model": "claude-haiku-4-5-20251001",
        "max_tokens": 2000,
    },
    "pipeline_params": None,  # set by configure_pipeline()
}

# --- Pipeline snapshot & params ---
pipeline_config_full = await show_pipeline_snapshot(session)
pipeline_params = configure_pipeline(session, campaign_config)

In [ ]:
# @title Load data & evaluation context

train_data, index_terms = prepare_datasets(
    session.store,
    session.backend_id,
    excel_path=r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx",
)
session.index_terms = index_terms

baseline_ps, dataset, campaign_rounds, baseline_results = await prepare_scoring_context(
    session,
    train_data,
    campaign_config,
    pipeline_params=pipeline_params,
)

In [ ]:
# @title Experiment dashboard
EXPERIMENT_ID = None  # Set to hex ID to resume (e.g. '68e2c5')
pipeline_params = locals().get("pipeline_params")  # preserve on re-run

pipeline_params = show_experiment_dashboard(
    session=session,
    experiment_id=EXPERIMENT_ID,
    campaign_config=campaign_config,
    dataset=dataset,
    baseline_prompt_fields=campaign_rounds[0]["prompt_fields"].model_dump()
    if campaign_rounds
    else None,
    pipeline_params=pipeline_params,
)

## 3. Explore

Exploration via **Smart Search** (scan advisor + sensitivity scan).

In [ ]:
# @title Task context

task_context = await decompose_task_context(TASK_DESCRIPTION, campaign_config, session)


## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [ ]:
# @title Feedback cycle preflight
show_feedback_preflight(
    campaign_rounds,
    dataset,
    campaign_config,
    session=session,
)


In [ ]:
# @title Run optimization (feedback cycle)
dev_reload()

campaign_rounds, _cycle_result = await run_optimization_notebook(
    campaign_rounds,
    dataset,
    campaign_config,
    session=session,
    experiment_id=EXPERIMENT_ID,
    task_context=task_context,
)


In [ ]:
# @title 5. Results — summary, save, sync
show_campaign_summary(campaign_rounds)
show_flip_tracking(campaign_rounds)
show_lineage_chain(campaign_rounds)

# --- Persist (T2: below the fold) ---
save_campaign_winner(
    campaign_rounds,
    campaign_config,
    session.store,
    session.backend_id,
    campaign_id=EXPERIMENT_ID,
)
sync_langfuse(
    session.store,
    session.backend_id,
    dataset_name="termnorm_ground_truth",
)